# SMS Spam Classification with Transformers

A transformer-based text classification workflow comparing zero-shot classification, fine-tuned BERT, few-shot BERT, and Flan-T5 generative prompting on the SMS Spam dataset.

## What this notebook demonstrates

- Load and preprocess SMS ham/spam messages.
- Evaluate zero-shot transformer classification.
- Fine-tune BERT with full supervision.
- Simulate few-shot learning with limited labeled examples.
- Compare generative Flan-T5 zero-shot and few-shot prompting.



# Objective:
This project is designed to provide comprehensive hands-on experience with **Transformer-based models** for text classification. The primary goal is to compare different transfer learning strategies, ranging from **Zero-Shot Learning** (inference without training) to **Fine-Tuning** and **Few-Shot Learning** using pre-trained language models like **BERT** and **Flan-T5**. We evaluate the effectiveness of these approaches in detecting SMS Spam using metrics such as Accuracy, F1-score, and Confusion Matrices.
## Dataset: [SMS Spam Collection Dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?resource=download)
## Tasks Covered:
1. **Task 1: Zero-Shot Classification** using a pre-trained BERT/BART variant (No training).
2. **Task 2: Fine-Tuning** a BERT model on the full dataset (Transfer Learning).
3. **Task 3: Few-Shot Learning** by fine-tuning BERT on a small subset of data.
4. **Task 4: Generative Classification** using an LLM (Flan-T5) with Zero-Shot & Few-Shot Prompting.


# Data Preparation and Preprocessing
In this section, we load the SMS Spam Collection dataset and perform necessary preprocessing steps. The data is cleaned, labels are encoded (ham=0, spam=1), and the dataset is split into training, validation, and testing sets using stratified sampling to maintain class balance


In [ ]:
!pip install -U transformers scikit-learn pandas seaborn matplotlib


In [ ]:
# Install necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# Load the dataset
df = pd.read_csv('spam.csv', encoding='latin-1')
# Data Cleaning
# Keep only necessary columns
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
# Convert labels to numbers: ham=0, spam=1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
# Check data shape
print(f"Total messages: {len(df)}")
print(df.head())
# Split Data (70% Train, 15% Val, 15% Test)
# We use stratify to keep the same percentage of spam in all sets
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['message'], df['label'], test_size=0.3, random_state=42, stratify=df['label']
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)
# Prepare numerical labels for later tasks
test_labels_num = test_labels.map({'ham': 0, 'spam': 1}).astype(int)
print(f"Train size: {len(train_texts)}")
print(f"Validation size: {len(val_texts)}")
print(f"Test size: {len(test_texts)}")


# Zero-Shot Classification (No Training)
Here, we employ a pre-trained Transformer model (RoBERTa/BERT variant) optimized for Natural Language Inference (NLI). We apply the model in a Zero-Shot setting to classify messages based on semantic meaning without any fine-tuning on our specific dataset. This establishes a baseline performance


The low accuracy (~29%) in Zero-Shot classification is likely due to the label name 'ham'. The NLI model interprets 'ham' as a food item, not as 'safe message'. Changing the label to 'non-spam' would significantly improve performance


In [ ]:
# Load a pipeline for zero-shot classification
# We use a BERT-based model suitable for this task
classifier = pipeline("zero-shot-classification", model="textattack/bert-base-uncased-MNLI", device=0)
candidate_labels = ["ham", "spam"]
# Run prediction on the test set
# We convert to list for faster processing
print("Classifying test set...")
results = classifier(list(test_texts), candidate_labels, truncation=True)
# Extract predicted labels
predicted_labels = [result['labels'][0] for result in results]
# Metrics
acc_task1 = accuracy_score(test_labels, predicted_labels)
print(f"Accuracy: {acc_task1:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, predicted_labels))
# Confusion Matrix
cm = confusion_matrix(test_labels, predicted_labels, labels=["ham", "spam"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.title("Zero-Shot Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


# Fine-Tuning BERT (Full Supervision)
In this task, we implement Transfer Learning by fine-tuning a standard bert-base-uncased model on the full training dataset. We add a classification head and train the model for 2 epochs to adapt its weights specifically for the SMS Spam detection task


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
# Tokenization
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize_data(texts):
    return tokenizer(list(texts), padding="max_length", truncation=True, max_length=128)
train_encodings = tokenize_data(train_texts)
val_encodings = tokenize_data(val_texts)
test_encodings = tokenize_data(test_texts)
# Create Dataset Class
class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)
# Convert labels to integers
train_labels_num = train_labels.map({'ham': 0, 'spam': 1}).astype(int)
val_labels_num = val_labels.map({'ham': 0, 'spam': 1}).astype(int)
# Create datasets
train_dataset = SpamDataset(train_encodings, train_labels_num)
val_dataset = SpamDataset(val_encodings, val_labels_num)
test_dataset = SpamDataset(test_encodings, test_labels_num)
# Model Setup
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }
# Training Arguments
training_args = TrainingArguments(
    output_dir='./results_task2',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_steps=50
)
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)
# Train
trainer.train()
# Evaluate on Test Set
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
# Results
acc_task2 = accuracy_score(test_labels_num, preds)
print(f"Accuracy: {acc_task2:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels_num, preds, target_names=['ham', 'spam']))
# Confusion Matrix
cm = confusion_matrix(test_labels_num, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.title("Fine-Tuned BERT Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


# Few-Shot Learning (Low-Resource Simulation)
This task evaluates the model's ability to learn from limited data. We simulate a low-resource scenario by fine-tuning a fresh BERT model on a very small, balanced subset of the training data (e.g., only 20 samples). This demonstrates the model's adaptability with minimal supervision


In [ ]:
# Set seed for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
# Select a small balanced subset (10 ham, 10 spam)
train_df = pd.DataFrame({'text': train_texts, 'label': train_labels})
# Group by label and sample 10 from each
few_shot_df = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(10, random_state=42))
print(f"Few-shot training samples: {len(few_shot_df)}")
# Prepare data for the few-shot model
few_train_encodings = tokenize_data(few_shot_df['text'])
few_train_labels = few_shot_df['label'].map({'ham': 0, 'spam': 1}).astype(int)
few_train_dataset = SpamDataset(few_train_encodings, few_train_labels)
# Load a fresh model
model_few = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
# Training arguments
training_args_few = TrainingArguments(
    output_dir='./results_task3',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    logging_steps=10,
    learning_rate=5e-5
)
trainer_few = Trainer(
    model=model_few,
    args=training_args_few,
    train_dataset=few_train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)
# Train
trainer_few.train()
# Evaluate
predictions_few = trainer_few.predict(test_dataset)
preds_few = np.argmax(predictions_few.predictions, axis=1)
# Results
acc_task3 = accuracy_score(test_labels_num, preds_few)
print(f"Accuracy: {acc_task3:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels_num, preds_few, target_names=['ham', 'spam']))
# Confusion Matrix
cm = confusion_matrix(test_labels_num, preds_few)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.title("Few-Shot BERT Confusion Matrix")
plt.show()


# Generative Classification using LLMs (Flan-T5)
In this section, we utilize a Generative Model (google/flan-t5-large) instead of a traditional classifier. We apply Prompt Engineering techniques to classify messages:
1.   Zero-Shot Prompting: Asking the model directly without examples.
2.   Few-Shot Prompting: Providing the model with a few examples (In-Context Learning) to guide its decision.


In [ ]:
# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1
# Load Flan-T5 model
generator = pipeline("text2text-generation", model="google/flan-t5-large", device=device)
# Helper function to clean output
def clean_prediction(text):
    text = text.lower()
    if "spam" in text:
        return 1
    elif "ham" in text:
        return 0
    else:
        return 0 # Default to ham if unclear
# Use a sample of 200 for speed
sample_n = min(200, len(test_texts))
sample_indices = random.sample(range(len(test_texts)), sample_n)
sampled_texts = [test_texts.iloc[i] for i in sample_indices]
sampled_labels = [test_labels_num.iloc[i] for i in sample_indices]
# Zero-Shot Prompting
print("Running Zero-Shot...")
prompts_zero = [f"Classify this SMS as ham or spam. Answer one word.\nMessage: {msg}\nLabel:" for msg in sampled_texts]
outputs_zero = generator(prompts_zero, max_new_tokens=5)
preds_zero = [clean_prediction(out['generated_text']) for out in outputs_zero]
acc_task4_zero = accuracy_score(sampled_labels, preds_zero)
print(f"Zero-Shot Accuracy: {acc_task4_zero:.4f}")
print(classification_report(sampled_labels, preds_zero, target_names=['ham', 'spam']))
# Plot Zero-Shot Matrix
cm_z = confusion_matrix(sampled_labels, preds_zero)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_z, annot=True, fmt='d', cmap='Blues')
plt.title("Task 4: Zero-Shot Confusion Matrix")
plt.show()
# Few-Shot Prompting
print("Running Few-Shot...")
# Get examples
ex_ham = train_texts[train_labels=='ham'].iloc[0]
ex_spam = train_texts[train_labels=='spam'].iloc[0]
prompts_few = []
for msg in sampled_texts:
    prompt = f"""Classify as ham or spam.
Example: {ex_ham} -> ham
Example: {ex_spam} -> spam
Message: {msg} ->"""
    prompts_few.append(prompt)
outputs_few = generator(prompts_few, max_new_tokens=5)
preds_few_gen = [clean_prediction(out['generated_text']) for out in outputs_few]
acc_task4_few = accuracy_score(sampled_labels, preds_few_gen)
print(f"Few-Shot Accuracy: {acc_task4_few:.4f}")
print(classification_report(sampled_labels, preds_few_gen, target_names=['ham', 'spam']))
# Plot Few-Shot Matrix
cm_f = confusion_matrix(sampled_labels, preds_few_gen)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_f, annot=True, fmt='d', cmap='Oranges')
plt.title("Task 4: Few-Shot Confusion Matrix")
plt.show()


# Final Performance Comparison
Finally, we visualize and compare the accuracy of all implemented methods (Zero-Shot, Fine-Tuned, Few-Shot, and Generative approaches) to analyze the trade-offs between training effort, data availability, and model performance.


In [ ]:
# Comparison of all tasks
methods = ['Zero-Shot', 'Fine-Tuned', 'Few-Shot', 'Gen Zero', 'Gen Few']
scores = [acc_task1, acc_task2, acc_task3, acc_task4_zero, acc_task4_few]
plt.figure(figsize=(10, 6))
sns.barplot(x=methods, y=scores, palette='viridis')
plt.title("Comparison of Accuracy Across All Tasks")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
# Add scores on top of bars
for i, v in enumerate(scores):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center')
plt.show()


# Transformer-Based SMS Spam Classification
## Objective
This part compares multiple transformer-based strategies for SMS spam detection using **Accuracy, F1-score, and Confusion Matrices**:
1) Zero-Shot BERT (no training)  
2) Fine-Tuned BERT (full supervision)  
3) Few-Shot BERT (limited samples)  
4) Generative classification with Flan-T5 (zero-shot & few-shot prompting)
## Dataset
We used the **SMS Spam Collection Dataset** (ham/spam).  
A stratified split was applied:
- 70% Train, 15% Validation, 15% Test  
**Total:** 5572 messages  
**Test:** 836 messages  
Labels encoded as: ham=0, spam=1.
## Method Summary
- **Task 1:** Zero-shot inference using `textattack/bert-base-uncased-MNLI`.  
- **Task 2:** Fine-tuned `bert-base-uncased` for 2 epochs.  
- **Task 3:** Few-shot BERT trained on **20 balanced samples** (10 ham + 10 spam) for 10 epochs.  
  **Note on Few-Shot Sample Size:**  
  A balanced set of 20 samples was chosen to simulate a true low-resource scenario. The goal is to evaluate how well BERT adapts with minimal supervision rather than to maximize accuracy. Larger sample sizes generally improve stability but make the scenario less representative of real few-shot learning.
- **Task 4:** `google/flan-t5-large` used via prompting; evaluated on a **sample of 200** test messages for speed.  
  **Note on Evaluation Subset:**  
  Due to the high computational cost of running Flan-T5-Large on the full test set in Google Colab, a subset of 200 messages was used for evaluation. This maintains representativeness while keeping runtime manageable. Running on the full test set is possible but significantly slower and may exceed Colab's GPU limits.
## Results Overview
| Method | Test Size | Accuracy |
|-------|----------|----------|
| Task 1: BERT Zero-Shot | 836 | **0.2871** |
| Task 2: BERT Fine-Tuned | 836 | **0.9952** |
| Task 3: BERT Few-Shot | 836 | **0.9294** |
| Task 4: Flan-T5 Zero-Shot | 200 | **0.7650** |
| Task 4: Flan-T5 Few-Shot | 200 | **0.6600** |
Key observations:
- **Fine-tuning achieved the best performance**, showing the strongest benefit from task-specific supervision.  
- **Few-shot BERT performed strongly** despite training on only 20 examples, but with more false positives than full fine-tuning.  
- **Zero-shot BERT was weak**, largely because the label **“ham”** is not semantically clear to NLI models, causing excessive spam predictions.  
- **Flan-T5 prompting achieved moderate accuracy but very high spam recall**, meaning it rarely missed spam but misclassified more ham as spam.  
- Few-shot prompting **did not outperform zero-shot** in our setup, likely due to prompt sensitivity and example selection.
## Conclusion
Overall, results confirm that:
- **Supervised fine-tuning is the most reliable approach** for SMS spam classification.  
- **Few-shot fine-tuning is a strong alternative** when data is limited.  
- **Generative prompting can be useful for quick deployment** without training, especially when prioritizing spam recall.  
- **Zero-shot NLI with “ham/spam” labels is limited** and may improve with clearer label wording (e.g., “not spam” vs “spam”).
